<a href="https://colab.research.google.com/github/abdullahpi912/InternshipProgress/blob/main/Phase-2/Day15/HyperparameterTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("Crop_recommendation.csv")
X = df.drop("label", axis=1)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [2]:
default_model = KNeighborsClassifier()   # uses default n_neighbors=5
default_model.fit(X_train, y_train)
default_pred = default_model.predict(X_test)
default_accuracy = accuracy_score(y_test, default_pred)

print("Default K:", default_model.n_neighbors)
print("Default Accuracy:", default_accuracy)

Default K: 5
Default Accuracy: 0.9704545454545455


In [3]:
# Tests every K from 1 to 20, one at a time, using 5-fold CV on each
param_grid = {"n_neighbors": list(range(1, 21))}

grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5)
grid_search.fit(X_train, y_train)

print("Best Hyperparameter:", grid_search.best_params_)
print("Best Cross Validation Score:", grid_search.best_score_)

Best Hyperparameter: {'n_neighbors': 1}
Best Cross Validation Score: 0.9823863636363637


In [4]:
best_k_grid = grid_search.best_params_["n_neighbors"]

grid_model = KNeighborsClassifier(n_neighbors=best_k_grid)
grid_model.fit(X_train, y_train)
grid_pred = grid_model.predict(X_test)
grid_accuracy = accuracy_score(y_test, grid_pred)

print("Tuned Accuracy (GridSearchCV):", grid_accuracy)

Tuned Accuracy (GridSearchCV): 0.9659090909090909


In [5]:
# Only tries a random sample of K values (n_iter=10) instead of testing all 20
random_search = RandomizedSearchCV(KNeighborsClassifier(), param_grid, cv=5, n_iter=10, random_state=42)
random_search.fit(X_train, y_train)

print("Best Hyperparameter:", random_search.best_params_)
print("Best Cross Validation Score:", random_search.best_score_)

best_k_random = random_search.best_params_["n_neighbors"]
random_model = KNeighborsClassifier(n_neighbors=best_k_random)
random_model.fit(X_train, y_train)
random_pred = random_model.predict(X_test)
random_accuracy = accuracy_score(y_test, random_pred)

print("Tuned Accuracy (RandomizedSearchCV):", random_accuracy)

Best Hyperparameter: {'n_neighbors': 1}
Best Cross Validation Score: 0.9823863636363637
Tuned Accuracy (RandomizedSearchCV): 0.9659090909090909


In [6]:
comparison = pd.DataFrame({
    "Method": ["Default", "GridSearchCV", "RandomizedSearchCV"],
    "Best K": [default_model.n_neighbors, best_k_grid, best_k_random],
    "Accuracy": [default_accuracy, grid_accuracy, random_accuracy],
})
print(comparison)


               Method  Best K  Accuracy
0             Default       5  0.970455
1        GridSearchCV       1  0.965909
2  RandomizedSearchCV       1  0.965909


In [7]:
best_row = comparison.loc[comparison["Accuracy"].idxmax()]

print(f"Conclusion: {best_row['Method']} performed best with K={best_row['Best K']}")
print(f"and an accuracy of {best_row['Accuracy']:.4f}.")
print("GridSearchCV and RandomizedSearchCV both improved on the default model by")
print("systematically testing multiple K values with cross validation instead of")
print("guessing one value, removing the risk of an unlucky default choice.")

Conclusion: Default performed best with K=5
and an accuracy of 0.9705.
GridSearchCV and RandomizedSearchCV both improved on the default model by
systematically testing multiple K values with cross validation instead of
guessing one value, removing the risk of an unlucky default choice.


In [8]:
print("Q1. What is Hyperparameter Tuning?")
print("The process of testing different hyperparameter values (like K in KNN) to find")
print("the combination that gives the best model performance.\n")

print("Q2. Difference between Parameters and Hyperparameters?")
print("Parameters are learned by the model automatically during training (like the")
print("splits in a Decision Tree). Hyperparameters are set manually before training")
print("(like K in KNN) and control how the model learns.\n")

print("Q3. What is GridSearchCV?")
print("A method that tries EVERY combination of specified hyperparameter values,")
print("using cross validation to score each one, then picks the best.\n")

print("Q4. What is RandomizedSearchCV?")
print("A method that tries only a random sample of hyperparameter combinations")
print("instead of all of them, trading a little accuracy for much faster search.\n")

print("Q5. Difference between GridSearchCV and RandomizedSearchCV?")
print("GridSearchCV is exhaustive and slower but guaranteed to check every option.")
print("RandomizedSearchCV is faster and scales better to large search spaces, but")
print("might miss the single best combination since it doesn't check everything.\n")

print("Q6. Why is Cross Validation used in Hyperparameter Tuning?")
print("It scores each hyperparameter choice across multiple data splits instead of")
print("just one, so the best choice isn't picked based on a lucky single split.\n")

print("Q7. Purpose of best_params_ and best_score_?")
print("best_params_ shows which hyperparameter combination performed best during the")
print("search. best_score_ shows the mean cross validation accuracy that combination")
print("achieved, letting you judge how good that best choice actually was.")

Q1. What is Hyperparameter Tuning?
The process of testing different hyperparameter values (like K in KNN) to find
the combination that gives the best model performance.

Q2. Difference between Parameters and Hyperparameters?
Parameters are learned by the model automatically during training (like the
splits in a Decision Tree). Hyperparameters are set manually before training
(like K in KNN) and control how the model learns.

Q3. What is GridSearchCV?
A method that tries EVERY combination of specified hyperparameter values,
using cross validation to score each one, then picks the best.

Q4. What is RandomizedSearchCV?
A method that tries only a random sample of hyperparameter combinations
instead of all of them, trading a little accuracy for much faster search.

Q5. Difference between GridSearchCV and RandomizedSearchCV?
GridSearchCV is exhaustive and slower but guaranteed to check every option.
RandomizedSearchCV is faster and scales better to large search spaces, but
might miss the si